# 评测集长什么样，ABCD 在训练里长什么样

对比四份数据的文本格式，不跑主模型：

| 名称 | 文件 | 用途 |
|---|---|---|
| VSI | `vsibench_manifest_32f.jsonl` | 评测，候选余弦匹配 |
| MMSI | `mmsibench_manifest.jsonl` | 评测，候选余弦匹配 |
| SPAR-select | `spar_sftqa_eval.jsonl` | 域内评测（select 题） |
| SPAR-train | `spar_sftqa_sftvljepa_uni8.jsonl` | 训练（sentence + fill 题） |

评测打分方式（`infer_gvjepa.py::_score_candidate_batch`）：`pred = predictor(图, query)`，
每个候选字符串过 Y-encoder，取余弦最大的候选。所以候选字符串的**字面形式**直接决定分数。

In [ ]:
import json, os, random, re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

ROOT = Path(os.environ.get("VJEPA_ROOT", "/glob/g01-cache/pf/Yushuo/vjepa201"))
SETS = {
    "VSI":         ROOT / "data/vsibench_manifest_32f.jsonl",
    "MMSI":        ROOT / "data/mmsibench_manifest.jsonl",
    "SPAR-select": ROOT / "data/spar_sftqa_eval.jsonl",
    "SPAR-train":  ROOT / "data/spar_sftqa_sftvljepa_uni8.jsonl",
}
RAW_VSI  = ROOT / "source_data/vsibench/test.jsonl"
RAW_MMSI = ROOT / "data/mmsibench_raw.jsonl"
RAW_SPAR = ROOT / "source_data/spar"
MAX_ROWS = 50_000     # 每份最多读多少行（大文件用蓄水池抽样）
SEED = 0

In [ ]:
def load(path, max_rows=MAX_ROWS, seed=SEED):
    rng, keep, seen = random.Random(seed), [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            seen += 1
            if len(keep) < max_rows:
                keep.append(line)
            else:
                j = rng.randrange(seen)
                if j < max_rows:
                    keep[j] = line
    return [json.loads(l) for l in keep], seen

data = {}
for name, path in SETS.items():
    if not path.exists():
        print(f"[skip] {name}: {path} 不存在"); continue
    data[name], total = load(path)
    print(f"{name:12s} 文件 {total:>9,} 行，读入 {len(data[name]):>7,}  字段={sorted(data[name][0].keys())}")

## 1. 概览：题型、图像数、候选数、随机水平、答案位置

- **chance** = 平均 `1/候选数`。
- **答案位置**：正确答案在候选列表里的下标分布。偏斜说明存在"总选某一项"的捷径。

In [ ]:
def qtype(r):
    return r.get("question_type") or r.get("type") or "<none>"

for name, rows in data.items():
    n = len(rows)
    cands = [r.get("candidates") or [] for r in rows]
    has = [c for c in cands if c]
    print(f"\n══ {name}  (n={n:,}) ══")
    print("图像数分布 :", dict(sorted(Counter(len(r.get('images') or [r.get('image')]) for r in rows).items())))
    print(f"有候选的行 : {len(has):,} ({100*len(has)/n:.1f}%)")
    if has:
        print("候选数分布 :", dict(sorted(Counter(len(c) for c in has).items())))
        print(f"chance     : {100*np.mean([1/len(c) for c in has]):.1f}%")
        pos = Counter(r["candidates"].index(r["target"]) if r["target"] in r["candidates"] else -1
                      for r in rows if r.get("candidates"))
        print("答案位置   :", {("ABCDEFGH"[k] if k >= 0 else "不在候选里"): f"{100*v/len(has):.1f}%"
                              for k, v in sorted(pos.items())})
    tc = Counter(qtype(r) for r in rows)
    print("题型       :")
    for t, k in tc.most_common(40):
        sub = [r for r in rows if qtype(r) == t and r.get("candidates")]
        ch = f"chance={100*np.mean([1/len(r['candidates']) for r in sub]):.0f}%" if sub else "无候选"
        print(f"    {t:38s} {k:>7,}  {ch}")

## 2. 每个题型的完整样例（query / candidates / target 原样打印）

In [ ]:
N_PER_TYPE = 2

def show(r, indent="  "):
    print(f"{indent}query     : {r.get('query', '')!r}")
    if r.get("candidates"):
        for i, c in enumerate(r["candidates"]):
            mark = "  <== target" if c == r["target"] else ""
            print(f"{indent}cand[{i}]   : {c!r}{mark}")
    print(f"{indent}target    : {r['target']!r}")

rng = random.Random(SEED)
for name, rows in data.items():
    by = defaultdict(list)
    for r in rows:
        by[qtype(r)].append(r)
    print(f"\n{'═'*30} {name} {'═'*30}")
    for t, rs in sorted(by.items(), key=lambda kv: -len(kv[1])):
        print(f"\n── {name} / {t}  (n={len(rs):,})")
        for r in rng.sample(rs, k=min(N_PER_TYPE, len(rs))):
            show(r); print()

## 3. 格式对照表

逐项量化评测候选和训练 target 的字面差异。重点看前三列：训练 target 没有字母前缀、query 里没有选项，评测如果有，就是分布偏移。

In [ ]:
PREFIX_RE   = re.compile(r"^\s*[A-H]\s*[\.:\)]\s+")
OPT_LINE_RE = re.compile(r"(?m)^\s*[A-H]\s*[\.:\)]\s+\S|(?<![A-Za-z])[A-D]:\s*\S")
OPT_WORD_RE = re.compile(r"\boptions?\b|\bchoices?\b|choose", re.I)
NUM_RE      = re.compile(r"^[-+]?\d+(?:\.\d+)?\s*[a-z°%]*\.?$", re.I)
LETTER_RE   = re.compile(r"^[A-H]\.?$")

def strip_prefix(s):
    return PREFIX_RE.sub("", s)

def fmt_stats(rows):
    n = len(rows)
    texts = [c for r in rows for c in (r.get("candidates") or [])]
    tg = [r["target"] for r in rows]
    q = [r.get("query", "") for r in rows]
    f = lambda xs, pred: 100 * sum(pred(x) for x in xs) / max(len(xs), 1)
    dup_after = sum(len(set(map(strip_prefix, r["candidates"]))) < len(r["candidates"])
                    for r in rows if r.get("candidates"))
    return {
        "target带字母前缀%":  f(tg, lambda s: bool(PREFIX_RE.match(s))),
        "候选带字母前缀%":    f(texts, lambda s: bool(PREFIX_RE.match(s))) if texts else float("nan"),
        "query内含选项%":     f(q, lambda s: len(OPT_LINE_RE.findall(s)) >= 2),
        "query提到options%":  f(q, lambda s: bool(OPT_WORD_RE.search(s))),
        "target是单个字母%":  f(tg, lambda s: bool(LETTER_RE.match(s.strip()))),
        "target是纯数字%":    f(tg, lambda s: bool(NUM_RE.match(strip_prefix(s).strip()))),
        "target词数中位":     float(np.median([len(strip_prefix(s).split()) for s in tg])),
        "query词数中位":      float(np.median([len(s.split()) for s in q])),
        "去前缀后候选重复行": dup_after,
    }

stats = {name: fmt_stats(rows) for name, rows in data.items()}
names = list(stats)
print(f"{'':22s}" + "".join(f"{n:>14s}" for n in names))
for k in next(iter(stats.values())):
    print(f"{k:22s}" + "".join(f"{stats[n][k]:14.1f}" for n in names))

## 4. 原始数据 → manifest：转换时改了什么

VSI 和 MMSI 的 manifest 保留了 `"A. "` 前缀；SPAR select 在转换时去掉了前缀和选项行。

In [ ]:
def first_rows(path, n=3):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
            if len(out) >= n:
                break
    return out

if RAW_VSI.exists():
    print("══ VSI-Bench 原始 test.jsonl ══")
    for r in first_rows(RAW_VSI, 400):
        if r.get("options"):
            print({k: r[k] for k in ("id", "question_type", "question", "options", "ground_truth") if k in r})
            m = next((x for x in data.get("VSI", []) if x.get("id") == r.get("id")), None)
            print("  -> manifest:"); show(m, "     ") if m else print("     (不在已读入的 manifest 样本里)")
            break
    num = [r for r in first_rows(RAW_VSI, 400) if not r.get("options")][:2]
    print("\n  无选项的数值题（mc 模式下被丢弃，未评测）:")
    for r in num:
        print("   ", {k: r[k] for k in ("question_type", "question", "ground_truth") if k in r})
else:
    print("[skip] 没有", RAW_VSI)

if RAW_MMSI.exists():
    print("\n══ MMSI-Bench 原始 ══")
    r = first_rows(RAW_MMSI, 1)[0]
    print({k: (v if k != "images" else f"<{len(v)} images>") for k, v in r.items()})
    m = next((x for x in data.get("MMSI", []) if x.get("id") == r.get("id")), None)
    print("  -> manifest:"); show(m, "     ") if m else print("     (不在已读入的 manifest 样本里)")
else:
    print("[skip] 没有", RAW_MMSI)

## 5. ABCD 在训练里长什么样

SPAR 每个题型有三种问法：`sentence`、`fill`、`select`。`convert_spar_sftqa.py` 的划分是
**sentence + fill → 训练，select → 评测**。所以训练时模型从没见过选择题形式。

下面对同一题型并排打印三种问法的原始对话（`human` = query，`gpt` = target）。

In [ ]:
N_TYPES = 6

def raw_example(type_dir, kind):
    files = sorted((type_dir / kind).glob("*.jsonl")) if (type_dir / kind).is_dir() else []
    for fp in files:
        with open(fp, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    conv = r.get("conversations") or []
                    get = lambda role: next((c.get("value", "") for c in conv if c.get("from") == role), "")
                    imgs = r.get("image") or r.get("sorted_image") or []
                    return get("human"), get("gpt"), (len(imgs) if isinstance(imgs, list) else 1)
    return None

type_dirs = sorted(RAW_SPAR.glob("*/qa_jsonl/train/*")) if RAW_SPAR.exists() else []
if not type_dirs:
    print("[skip] 没有", RAW_SPAR)
seen_types = set()
for d in type_dirs:
    if d.name in seen_types or len(seen_types) >= N_TYPES:
        continue
    seen_types.add(d.name)
    print(f"\n{'═'*20} {d.name}  ({d.parts[-4]}) {'═'*20}")
    for kind, role in (("sentence", "训练"), ("fill", "训练"), ("select", "评测")):
        ex = raw_example(d, kind)
        if ex is None:
            print(f"  [{kind:8s} -> {role}] (无)"); continue
        q, a, n_img = ex
        print(f"  [{kind:8s} -> {role}]  原始图像数={n_img}")
        print(f"      human: {q!r}")
        print(f"      gpt  : {a!r}")

In [ ]:
# 训练 manifest 里是否混进了选择题形式
if "SPAR-train" in data:
    tr = data["SPAR-train"]
    with_opts = [r for r in tr if len(OPT_LINE_RE.findall(r.get("query", ""))) >= 2]
    letter_t = [r for r in tr if LETTER_RE.match(r["target"].strip())]
    print(f"SPAR-train: query 内含选项的行 {len(with_opts):,}/{len(tr):,}，target 是单个字母的行 {len(letter_t):,}/{len(tr):,}")
    for r in (with_opts[:2] + letter_t[:2]):
        show(r); print()
    print("target 首词分布（看训练 target 以什么开头）:")
    for w, k in Counter((r["target"].split() or [""])[0].lower() for r in tr).most_common(25):
        print(f"    {100*k/len(tr):5.1f}%  {w!r}")

## 6. 字母前缀对 Y-encoder 的影响（可选，需要 GPU 或耐心）

对每道题的候选集算三个余弦量：

- **同内容换字母**：`"A. left"` vs `"B. left"`。越接近 1，说明字母几乎不影响向量。
- **同字母换内容**：`"A. left"` vs `"A. right"`（即带前缀的候选两两之间）。
- **无前缀**：`"left"` vs `"right"`。

如果"同字母换内容"明显高于"无前缀"，说明前缀把候选向量挤到了一起，候选之间更难区分。
`CKPT=None` 用预训练的 embeddinggemma + mean-pool；填训练好的 checkpoint 则加载其中的 `y_encoder` 和 `y_proj`，与评测一致。

In [ ]:
RUN_ENCODER = False
CKPT = None            # 例如 ROOT / "outputs/sft_from_vljepa_ckpt_vggt/final.pt"
Y_NAME = "google/embeddinggemma-300m"
N_ROWS = 300
MAX_LEN = 256

if RUN_ENCODER:
    import torch, torch.nn.functional as F
    from transformers import AutoModel, AutoTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(Y_NAME, cache_dir=str(ROOT / "ckpts"))
    enc = AutoModel.from_pretrained(Y_NAME, cache_dir=str(ROOT / "ckpts")).to(dev).eval()
    proj = None
    if CKPT is not None:
        sd = torch.load(CKPT, map_location="cpu")["model"]
        enc.load_state_dict({k[len("y_encoder."):]: v for k, v in sd.items() if k.startswith("y_encoder.")})
        w, b = sd["y_proj.weight"], sd["y_proj.bias"]
        proj = torch.nn.Linear(w.shape[1], w.shape[0]); proj.load_state_dict({"weight": w, "bias": b}); proj.to(dev)

    @torch.no_grad()
    def embed(texts, bs=64):
        out = []
        for i in range(0, len(texts), bs):
            t = tok(texts[i:i + bs], max_length=MAX_LEN, padding=True, truncation=True, return_tensors="pt").to(dev)
            hs = enc(**t).last_hidden_state
            m = t["attention_mask"].unsqueeze(-1).float()
            pooled = (hs * m).sum(1) / m.sum(1).clamp_min(1)       # 与 FusionGVJEPA.encode_target 相同
            out.append(F.normalize(proj(pooled) if proj is not None else pooled, dim=-1).float().cpu())
        return torch.cat(out)

    def offdiag_mean(e):
        s = e @ e.T
        n = s.shape[0]
        return ((s.sum() - s.diag().sum()) / (n * (n - 1))).item()

    for name in ("VSI", "MMSI", "SPAR-select"):
        rows = [r for r in data.get(name, []) if r.get("candidates") and len(r["candidates"]) >= 2]
        rows = random.Random(SEED).sample(rows, k=min(N_ROWS, len(rows)))
        if not rows:
            continue
        with_p, no_p, swap = [], [], []
        for r in rows:
            bare = [strip_prefix(c) for c in r["candidates"]]
            lettered = [f"{'ABCDEFGH'[i]}. {b}" for i, b in enumerate(bare)]
            e_l, e_b = embed(lettered), embed(bare)
            with_p.append(offdiag_mean(e_l)); no_p.append(offdiag_mean(e_b))
            e_sw = embed([f"B. {bare[0]}"])                          # 同内容换字母
            swap.append((e_l[0] @ e_sw[0]).item())
        print(f"{name:12s} n={len(rows)}  同内容换字母={np.mean(swap):.3f}  "
              f"同字母换内容={np.mean(with_p):.3f}  无前缀={np.mean(no_p):.3f}")
else:
    print("RUN_ENCODER=False，跳过。")

## 7. 看图：随机一道评测题的全部帧

In [ ]:
SHOW_IMAGES = True
SHOW_SET = "MMSI"
MAX_IMGS = 8

if SHOW_IMAGES and SHOW_SET in data:
    import matplotlib.pyplot as plt
    from PIL import Image

    r = random.choice(data[SHOW_SET])
    imgs = r.get("images") or [r.get("image")]
    step = max(1, len(imgs) // MAX_IMGS)
    pick = imgs[::step][:MAX_IMGS]
    fig, axes = plt.subplots(1, len(pick), figsize=(3 * len(pick), 3), squeeze=False)
    for ax, p in zip(axes[0], pick):
        ax.imshow(Image.open(p)); ax.axis("off")
    plt.tight_layout(); plt.show()
    print(f"{SHOW_SET}  共 {len(imgs)} 张，显示 {len(pick)} 张")
    show(r)